# 02 — CNN-LSTM Training
Preprocess DAiSEE from Google Drive, train CNN-LSTM, save best checkpoint.

**Runtime:** GPU (T4 recommended) · **Estimated time:** ~3-4 h preprocessing + ~4-6 h training

### Before running
1. Runtime → Change runtime type → **GPU (T4)**
2. Run all cells top to bottom
3. When prompted, authorise Google Drive access

## 1 · Mount Google Drive + install dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q mediapipe opencv-python-headless scikit-learn pyyaml tqdm

## 2 · Clone / copy training scripts to Colab local disk

In [ ]:
import shutil, os
from pathlib import Path

DRIVE_REPO = '/content/drive/MyDrive/affectlearn-ml'
LOCAL_SCRIPTS = '/content/affectlearn-ml'

# Copy training scripts to local disk for faster imports
shutil.copytree(f'{DRIVE_REPO}/training', f'{LOCAL_SCRIPTS}/training', dirs_exist_ok=True)
shutil.copy(f'{DRIVE_REPO}/training/facial/config.yaml', '/content/config.yaml')

import sys
sys.path.insert(0, f'{LOCAL_SCRIPTS}/training/facial')
print('Scripts ready at', LOCAL_SCRIPTS)

## 3 · Verify paths and check GPU

In [ ]:
import torch, yaml

cfg = yaml.safe_load(open('/content/config.yaml'))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only!')
print('Config paths:')
for k, v in cfg['paths'].items():
    exists = Path(v).exists() if v else False
    print(f'  {k}: {v}  [{"OK" if exists else "NOT FOUND"}]')

## 4 · SMOKE TEST — preprocess 50 clips first
This eyeballs the face-crop output before committing 3-4 hours to full preprocessing.
**Check the sample images below — if faces look wrong, stop here and debug.**

In [ ]:
import sys
sys.path.insert(0, '/content/affectlearn-ml/training/facial')

from preprocess import run_preprocessing

run_preprocessing(
    out_dir='/content/daisee_smoke',
    zip_path=cfg['paths']['daisee_zip'],
    splits=('Train',),
    smoke_n=50,
)
print('Smoke test done.')

In [ ]:
# Visualise 8 sample face crops to verify preprocessing quality
import numpy as np
import matplotlib.pyplot as plt

MEAN = np.array([0.485, 0.456, 0.406])
STD  = np.array([0.229, 0.224, 0.225])

def denorm(chw):
    hwc = chw.transpose(1, 2, 0)
    return np.clip(hwc * STD + MEAN, 0, 1)

smoke_clips = sorted(Path('/content/daisee_smoke/Train').glob('*.npy'))[:8]
fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for i, npy in enumerate(smoke_clips):
    clip = np.load(npy)          # (16, 3, 96, 96)
    axes[0, i].imshow(denorm(clip[0]))   # first frame
    axes[1, i].imshow(denorm(clip[-1]))  # last frame
    axes[0, i].set_title(npy.stem[:8], fontsize=7)
    for ax in [axes[0,i], axes[1,i]]:
        ax.axis('off')
axes[0, 0].set_ylabel('Frame 0', fontsize=8)
axes[1, 0].set_ylabel('Frame 15', fontsize=8)
plt.suptitle('Smoke test — verify face crops look correct', fontsize=11)
plt.tight_layout()
plt.show()

## 5 · Full preprocessing (3-4 hours)
Only run this after the smoke test looks good.

> **Tip:** Preprocessed .npy files are saved to local Colab disk (`/content/daisee_preprocessed`).
> The script skips already-processed clips so it is safe to re-run if the session dies.

In [ ]:
from preprocess import run_preprocessing

run_preprocessing(
    out_dir=cfg['paths']['daisee_preprocessed'],
    zip_path=cfg['paths']['daisee_zip'],
    splits=('Train', 'Validation', 'Test'),
)
print('Full preprocessing complete.')

## 6 · Check label distribution

In [ ]:
import csv
import numpy as np
import matplotlib.pyplot as plt

labels_dir = Path(cfg['paths']['labels_dir'])
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (split, fname) in zip(axes, [
    ('Train',      'TrainLabels.csv'),
    ('Validation', 'ValidationLabels.csv'),
    ('Test',       'TestLabels.csv'),
]):
    with open(labels_dir / fname) as f:
        rows = list(csv.DictReader(f))
    eng = [int(r['Engagement']) for r in rows]
    counts = np.bincount(eng, minlength=4)
    ax.bar(['Very Low', 'Low', 'High', 'Very High'], counts, color=['#e74c3c','#e67e22','#2ecc71','#3498db'])
    ax.set_title(f'{split} ({len(rows)} clips)')
    ax.set_ylabel('Count')
    for i, c in enumerate(counts):
        ax.text(i, c + 5, f'{c/len(rows)*100:.1f}%', ha='center', fontsize=8)

plt.suptitle('DAiSEE Engagement label distribution', fontsize=12)
plt.tight_layout()
plt.show()

## 7 · Train CNN-LSTM

In [ ]:
# Run the training script directly
!cd /content/affectlearn-ml/training/facial && python train_cnn_lstm.py --config /content/config.yaml

## 8 · Evaluate on test set

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from torch.utils.data import DataLoader

from model   import build_model
from dataset import DAiSEEDataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

ckpt      = torch.load(cfg['paths']['checkpoints'] + '/cnn_lstm_best.pt', map_location=device)
model     = build_model(cfg['model']).to(device)
model.load_state_dict(ckpt['model_state'])
model.eval()

test_ds = DAiSEEDataset(
    cfg['paths']['daisee_preprocessed'],
    Path(cfg['paths']['labels_dir']) / 'TestLabels.csv',
    split='Test', target=cfg.get('target_affect', 'Engagement'),
)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2)

all_preds, all_labels = [], []
with torch.no_grad():
    for clips, labels in test_loader:
        preds = model(clips.to(device)).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

print(classification_report(
    all_labels, all_preds,
    target_names=['Very Low', 'Low', 'High', 'Very High']
))

cm = confusion_matrix(all_labels, all_preds)
ConfusionMatrixDisplay(cm, display_labels=['Very Low','Low','High','Very High']).plot(cmap='Blues')
plt.title('Test set confusion matrix')
plt.tight_layout()
plt.show()

from sklearn.metrics import f1_score
wf1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
print(f'\nWeighted F1 on test set: {wf1:.4f}  (target ≥ 0.60)')

## 9 · Copy checkpoint back to Google Drive

In [ ]:
import shutil

drive_models = '/content/drive/MyDrive/affectlearn-ml/models'
Path(drive_models).mkdir(parents=True, exist_ok=True)

for fname in ['cnn_lstm_best.pt', 'cnn_lstm_best.onnx']:
    src = Path(cfg['paths']['checkpoints']) / fname
    if src.exists():
        shutil.copy(src, f'{drive_models}/{fname}')
        print(f'Copied {fname} → Google Drive')
    else:
        print(f'Not found: {fname}')